<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "508"
iso_country = "Mozambique"
year = 2023
import_portfolio = "h6_ag6_by_sea_M.csv" # Imports, '_M' | Foreign Imports, '_FM' | Re-Imports, '_RM'
export_portfolio = "h6_ag6_by_sea_X.csv" # Exports, '_X' | Domestic Exports, '_DX' | Re-exports, '_RX'
gdp = 20.930*1e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    #"TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Mozambique': 

Number of trade records present in dataset: 14958
Number of partner countries present in dataset: 139
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$6761.7m
Total weight within the dataset: 8270.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,508,Mozambique,156,China,Import,Sea,870919,870919 | Vehicles; (not elect,8709,"8709 | Works trucks, self-p",87,87 | Vehicles; other than,4353,Fork-lift trucks; other works trucks whether o...,Transport Equipment,41,OME - Machinery and equipment n.e.c.,6000,4000.0,6000,u,2.0,7280.0,576.6,12702.0,7892.654442,4550.904551,"['Container', 'General cargo']","['Container', 'General cargo']",20.306742,20.273908,20.277134,20.277134,20.292013,20.292013,21.145101,35.879209,22.595471,21.507710,21.547977,22.075243,20.748507,34.538227,23.757627,22.278765,21.813627,22.696015,20.400680,38.224072,29.047812,26.021418,24.264831,23.551346,20.059271,40.797320,33.493476,31.274788,25.612338,25.061290,19.724304,42.331298,39.693724,41.479806,27.123364,26.183046,19.382627,40.266409,37.831287,41.791603,33.131222,26.615323,19.060763,36.256645,38.985328,40.427027,30.474323,26.892689,18.732565,34.022381,34.520279,39.794107,33.524854,27.067058,4353
1,1,508,Mozambique,156,China,Import,Sea,870990,870990 | Vehicles; parts of t,8709,"8709 | Works trucks, self-p",87,87 | Vehicles; other than,4357,Parts for the goods of subclasses 43510 to 43560,Transport Equipment,41,OME - Machinery and equipment n.e.c.,23000,17000.0,23000,kg,26920.0,26920.0,2210.3,12702.0,7892.654442,17445.134113,"['Container', 'Ro-Ro']","['Container', 'Ro-Ro']",69.652403,69.504344,69.519901,69.519901,69.586683,69.586683,73.032521,131.666320,79.057209,75.009502,75.241648,76.824262,71.433488,128.004646,84.104197,77.616389,75.669758,79.338101,70.031086,141.968267,105.009025,92.030736,84.878776,82.804574,68.654530,150.105404,123.196504,113.669548,90.660442,88.892391,67.303941,154.088729,149.143617,151.643149,96.756372,93.439453,65.926304,150.786661,141.523920,153.511450,119.192533,95.212553,64.628576,132.338263,139.659301,145.700454,104.610433,96.343077,63.305313,121.724972,123.065307,142.716810,120.202658,97.048455,4357


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [4]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $1183.0m (17.5%), ... 

                            cifvalue    vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
10 | Cereals               640215000  1.672790e+09      9.5     20.2
15 | Animal, vegetable or  318446000  3.573030e+08      4.7      4.3
03 | Fish and crustaceans   47752000  2.258482e+07      0.7      0.3
02 | Meat and edible meat   46429000  2.519270e+07      0.7      0.3
19 | Preparations of cere   28432000  1.018150e+07      0.4      0.1
20 | Preparations of vege   16534000  8.395700e+06      0.2      0.1
22 | Beverages, spirits a   15789000  2.525670e+07      0.2      0.3
21 | Miscellaneous edible   14886000  4.018276e+06      0.2      0.0
11 | Products of the mill   13642000  2.539091e+07      0.2      0.3
07 | Vegetables and certa    8487000  1.255408e+07      0.1      0.2
17 | Sugars and sugar con    6598000  5.205078e+06      0.1      0.1
1

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [5]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $845.5m (12.5%), ... 

                   cifvalue    vol_kg_vtw  pct_val  pct_vol
eora_desc                                                  
Agriculture       750669000  1.056971e+09     11.1     12.8
Food & Beverages   94064000  5.694067e+07      1.4      0.7
Fishing              750000  1.942052e+05      0.0      0.0



### 2.1.3 GTAP (16x headings)

In [6]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $819.2m (12.1%), ... 

                                cifvalue   vol_kg_vtw  pct_val  pct_vol
gtap_desc                                                              
PCR - Processed rice           345370000  599907690.0      5.1      7.3
VOL - Vegetable oils and fats  319532000  360429050.2      4.7      4.4
OFD - Food products n.e.c.      76591000   32636883.0      1.1      0.4
OMT - Meat products n.e.c.      50673000   27491371.2      0.7      0.3
V_F - Vegetables, fruit, nuts    8511000   12500336.4      0.1      0.2
GRO - Cereal grains n.e.c.       7069000   29240918.5      0.1      0.4
MIL - Dairy products             5942000    2637193.5      0.1      0.0
CMT - Bovine meat prods          2720000    1409748.0      0.0      0.0
OCR - Crops n.e.c.               1457000     293970.6      0.0      0.0
PFB - Plant-based fibers          469000     277531.1      0.0      0.0
PDR - Paddy rice                  419000     727803.0      0.0      0.

## 2.2 Comparison of results

In [7]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 17.5% 
GTAP 12.1% 
EORA 12.5%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [8]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [9]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_46400/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,508,Mozambique,156,China,Import,Sea,870919,870919 | Vehicles; (not elect,8709,"8709 | Works trucks, self-p",87,87 | Vehicles; other than,4353,Fork-lift trucks; other works trucks whether o...,Transport Equipment,41,OME - Machinery and equipment n.e.c.,6000,4000.0,6000,u,2.0,7280.0,576.6,12702.0,7892.654442,4550.904551,"['Container', 'General cargo']","['Container', 'General cargo']",20.306742,20.273908,20.277134,20.277134,20.292013,20.292013,21.145101,35.879209,22.595471,21.507710,21.547977,22.075243,20.748507,34.538227,23.757627,22.278765,21.813627,22.696015,20.400680,38.224072,29.047812,26.021418,24.264831,23.551346,20.059271,40.797320,33.493476,31.274788,25.612338,25.061290,19.724304,42.331298,39.693724,41.479806,27.123364,26.183046,19.382627,40.266409,37.831287,41.791603,33.131222,26.615323,19.060763,36.256645,38.985328,40.427027,30.474323,26.892689,18.732565,34.022381,34.520279,39.794107,33.524854,27.067058,4353,NaN
1,1,508,Mozambique,156,China,Import,Sea,870990,870990 | Vehicles; parts of t,8709,"8709 | Works trucks, self-p",87,87 | Vehicles; other than,4357,Parts for the goods of subclasses 43510 to 43560,Transport Equipment,41,OME - Machinery and equipment n.e.c.,23000,17000.0,23000,kg,26920.0,26920.0,2210.3,12702.0,7892.654442,17445.134113,"['Container', 'Ro-Ro']","['Container', 'Ro-Ro']",69.652403,69.504344,69.519901,69.519901,69.586683,69.586683,73.032521,131.666320,79.057209,75.009502,75.241648,76.824262,71.433488,128.004646,84.104197,77.616389,75.669758,79.338101,70.031086,141.968267,105.009025,92.030736,84.878776,82.804574,68.654530,150.105404,123.196504,113.669548,90.660442,88.892391,67.303941,154.088729,149.143617,151.643149,96.756372,93.439453,65.926304,150.786661,141.523920,153.511450,119.192533,95.212553,64.628576,132.338263,139.659301,145.700454,104.610433,96.343077,63.305313,121.724972,123.065307,142.716810,120.202658,97.048455,4357,NaN


## 3.3 Inspect the output

In [10]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $833.6m of $6761.7m (12.3%)
Volume: 1823.2 of 8270.6 million kilograms (22.0%)
Value-to-Weight: 0.46 from 0.82 US$ per kg (+-44.1%)
Average distance to market (volume-weighted): 6788.3 from 5626.4 nm (+20.6%)

            food_group  N_com   cifvalue    vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    336  401659000  6.396467e+08      5.9      7.7
1  Unprocessed Cereals     23  294426000  1.072155e+09      4.4     13.0
2        Fats and Oils     51   69415000  6.401186e+07      1.0      0.8
3        Meat and Prep     74   46100000  2.647438e+07      0.7      0.3
4           Vegetables    102    9409000  1.301223e+07      0.1      0.2
5      Sugar and Honey     39    6618000  5.210812e+06      0.1      0.1
6                Dairy     52    5699000  2.564566e+06      0.1      0.0
7               Fruits     29     288000  1.622920e+05      0.0      0.0



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [11]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,3.967506e+06,7.131958e+06,4.292021e+06,4.072154e+06,4.084780e+06,4.171930e+06,3.658294e+06,8.348347e+06,8.076856e+06,8.217331e+06,5.252977e+06,5.068287e+06,3.442483e+06,6.600983e+06,6.676149e+06,7.737012e+06,6.516700e+06,5.263079e+06
1,Fats and Oils,1.400206e+08,2.341638e+08,1.497271e+08,1.411828e+08,1.415863e+08,1.458921e+08,1.308448e+08,2.659008e+08,2.607516e+08,2.693820e+08,1.802018e+08,1.723042e+08,1.244400e+08,2.189012e+08,2.210237e+08,2.546878e+08,2.145194e+08,1.779385e+08
2,Fruits,3.689539e+05,6.650068e+05,3.993740e+05,3.789184e+05,3.800908e+05,3.880972e+05,3.400294e+05,7.781829e+05,7.532395e+05,7.659006e+05,4.887581e+05,4.719880e+05,3.198398e+05,6.148171e+05,6.215838e+05,7.208271e+05,6.071133e+05,4.902091e+05
3,Meat and Prep,8.892218e+07,1.603129e+08,9.625766e+07,9.132929e+07,9.161195e+07,9.353889e+07,8.194723e+07,1.876138e+08,1.815927e+08,1.846361e+08,1.178076e+08,1.137690e+08,7.707863e+07,1.482086e+08,1.498406e+08,1.737677e+08,1.463551e+08,1.181633e+08
4,Processed Cereals,1.505704e+09,2.712528e+09,1.629634e+09,1.546189e+09,1.550977e+09,1.583725e+09,1.387792e+09,3.174700e+09,3.072572e+09,3.124500e+09,1.994656e+09,1.925661e+09,1.305489e+09,2.508525e+09,2.536375e+09,2.940864e+09,2.476975e+09,1.999917e+09
5,Sugar and Honey,1.312157e+07,2.328661e+07,1.415471e+07,1.342782e+07,1.346979e+07,1.377486e+07,1.212789e+07,2.727304e+07,2.634184e+07,2.687660e+07,1.732470e+07,1.664824e+07,1.143471e+07,2.166143e+07,2.193786e+07,2.535670e+07,2.135625e+07,1.727400e+07
6,Unprocessed Cereals,3.371391e+09,6.077680e+09,3.649449e+09,3.462597e+09,3.473314e+09,3.546397e+09,3.106983e+09,7.112750e+09,6.884447e+09,6.999903e+09,4.466540e+09,4.313275e+09,2.922423e+09,5.618962e+09,5.680881e+09,6.587908e+09,5.548655e+09,4.479842e+09
7,Vegetables,4.038402e+07,7.266604e+07,4.369659e+07,4.145844e+07,4.158693e+07,4.246995e+07,3.723001e+07,8.505332e+07,8.229857e+07,8.371246e+07,5.348111e+07,5.161522e+07,3.502825e+07,6.723050e+07,6.798648e+07,7.881159e+07,6.637623e+07,5.360210e+07


### 3.3.2 Total FEM costs by scenario (% cif)

In [12]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.56,0.06,0.02,0.02,0.04,0.82,0.78,0.80,0.28,0.25,0.55,0.57,0.75,0.54,0.32
1,Fats and Oils,1.36,0.14,0.02,0.02,0.08,1.95,1.87,2.00,0.71,0.60,1.36,1.39,1.88,1.30,0.77
2,Fruits,1.03,0.11,0.03,0.04,0.07,1.52,1.43,1.48,0.52,0.46,1.02,1.05,1.39,1.00,0.59
3,Meat and Prep,1.55,0.16,0.05,0.06,0.10,2.29,2.16,2.23,0.78,0.69,1.54,1.58,2.10,1.50,0.89
4,Processed Cereals,3.00,0.31,0.10,0.11,0.19,4.45,4.19,4.32,1.51,1.34,3.00,3.06,4.07,2.92,1.73
5,Sugar and Honey,1.54,0.16,0.05,0.05,0.10,2.29,2.15,2.23,0.79,0.68,1.55,1.59,2.10,1.50,0.88
6,Unprocessed Cereals,9.19,0.94,0.31,0.35,0.59,13.61,12.83,13.22,4.62,4.10,9.16,9.37,12.45,8.92,5.29
7,Vegetables,3.43,0.35,0.11,0.13,0.22,5.08,4.79,4.94,1.73,1.53,3.42,3.50,4.65,3.33,1.97
